# Day 2: Word Embeddings and Semantic Text Representation

Everything so far (CountVectorizer, TF-IDF) treats each word as a totally independent, unrelated symbol -- "stock" and "shares" are just two different columns with no built-in relationship, even though they mean similar things. Word embeddings fix that: each word becomes a dense vector positioned in space so that semantically similar words end up close together.

Using pretrained GloVe vectors here, not training our own -- 278 rows is nowhere near enough data to learn meaningful word relationships from scratch. GloVe was trained by Stanford on a huge combined corpus of Wikipedia + the Gigaword news archive (400,000 words, 100 numbers each).

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import gensim.downloader as api
from sklearn.feature_extraction.text import TfidfVectorizer

df = pd.read_csv("news_dataset.csv")

X_train_text, X_test_text, y_train_labels, y_test_labels = train_test_split(
    df["Title"], df["Category"], test_size=0.2, random_state=42
)

print("Train shape:", X_train_text.shape)
print("Test shape:", X_test_text.shape)

glove = api.load("glove-wiki-gigaword-100")
print("Vocabulary size:", len(glove.key_to_index))
print("Vector dimension:", glove.vector_size)

Train shape: (222,)
Test shape: (56,)


Vocabulary size: 400000
Vector dimension: 100


In [2]:
##semantic similarity
##most_similar will find the words whose vectors are closests to 'stocks', this shows that embedding actually captured real relationships and not memorization of pairs
similar_words = glove.most_similar("stock", topn=5)
print("Words most similar to 'stock':", similar_words)

##similarity() gives one number based on how close two words are, based on vetor positions
similarity_score = glove.similarity("stock", "shares")
print("Similarity between 'stock' and 'shares':", similarity_score)

##comparing a related pair against an unrelated pair, this is what CountVectorizer and TF-IDF lack
unrelated_score = glove.similarity("stock", "banana")
print("Similarity between 'stock' and 'banana':", unrelated_score)

Words most similar to 'stock': [('shares', 0.8525474667549133), ('stocks', 0.8309942483901978), ('market', 0.7991610765457153), ('exchange', 0.784952700138092), ('trading', 0.7632875442504883)]
Similarity between 'stock' and 'shares': 0.8525474
Similarity between 'stock' and 'banana': 0.15822881


"stock" is highly similar to "shares" (0.85), "stocks" (0.83), "market" (0.80) -- all genuinely related financial concepts the model found on its own from patterns in how these words get used across a massive amount of text. "stock" vs "banana" comes out at just 0.16. That's the whole point of semantic similarity, made concrete.

In [3]:
import numpy as np 
def get_sentence_embedding(text, model):
    ##simple split, not the full nltk preprocessing (day 1) to keep this step focused on the embedding itself
    words = text.lower().split()
    ##only keep vectors for words actually in GloVe's vocab, 'if word in model' this step skips anything that GloVe never saw during training (out-of-vocab)
    vectors = [model[word] for word in words if word in model]
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

sample = X_train_text.iloc[0]
embedding = get_sentence_embedding(sample, glove)
print("Headline:", sample)
print("Embedding shape:", embedding.shape)
print("First 5 values:", embedding[:5])

Headline: China's Xi to bring large CEO delegation on US visit, sources say
Embedding shape: (100,)
First 5 values: [-0.25916222  0.170759    0.3576308   0.00814938  0.0866853 ]


GloVe gives one vector per word, but classification needs one vector per *headline*. The standard simple fix is averaging every word's vector together -- a "sentence embedding". Shape comes out to (100,), confirming one combined vector for the whole headline instead of per word.

In [4]:
##train classifier and apply dataset
##converts every headline into its 100-number sentence embedding, the same function as the sample above
X_train_embeddings = np.array([get_sentence_embedding(text, glove) for text in X_train_text])
X_test_embeddings = np.array([get_sentence_embedding(text, glove) for text in X_test_text])

##should be (222,100) 222 headlines that are now 100-number vector instead of a huge jumble of words
print("Train embeddings shape:", X_train_embeddings.shape)

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

##some model type as day 1's TF-IDF/countvectorizer comparisons, for a fair three way comparison
model = LogisticRegression(max_iter=1000)
model.fit(X_train_embeddings, y_train_labels)
predictions = model.predict(X_test_embeddings)

print("Accuracy:", accuracy_score(y_test_labels, predictions))
print(classification_report(y_test_labels, predictions, zero_division=0))

Train embeddings shape: (222, 100)
Accuracy: 0.5892857142857143
              precision    recall  f1-score   support

    Business       0.55      0.94      0.70        17
      Energy       0.00      0.00      0.00         3
      Health       0.00      0.00      0.00         3
     Markets       0.67      0.60      0.63        20
    Politics       0.00      0.00      0.00         2
  Technology       0.62      0.45      0.53        11

    accuracy                           0.59        56
   macro avg       0.31      0.33      0.31        56
weighted avg       0.53      0.59      0.54        56



Side note: `X_train_embeddings` is only 100 columns wide vs. TF-IDF being 1148 -- this reduction in dimensionality is because of pretrained data doing the heavy lifting instead of counting words that happen to appear in our small dataset.

58.9% accuracy -- actually the *lowest* of everything tried so far, below both Day 1 results (CountVectorizer 71.4%, TF-IDF 62.5%). Plausible reason: averaging word vectors is a pretty crude way to build a sentence representation. It loses word order entirely (same as bag-of-words), but now it also blurs every word's individual identity into one blended average. A single highly distinctive word (like "vaccine" clearly signaling Health) gets diluted by averaging with 10 more generic words in the same headline, whereas CountVectorizer/TF-IDF let the classifier weight that one word heavily on its own, as a dedicated feature. There's also a mismatch in what GloVe was trained on (general Wikipedia/news) versus the narrow, specific distinctions our 6 categories need.

In [5]:
##IDF-weighted averaging
tfidf = TfidfVectorizer()
##reusing the TFIDF concept from day 1 to get the IDF weights per word, not builiding a classifier for it this time
tfidf.fit(X_train_text)
##maps each word tp fit the IDF score- how rare it is across the training set
idf_scores = dict(zip(tfidf.get_feature_names_out(), tfidf.idf_))

def get_weighted_sentence_embedding(text, model, idf_scores):
    words = text.lower().split()
    vectors = []
    weights = []
    for word in words:
        if word in model:
            ##default weight of 1.0 for any word the IDF never saw (avoids crash on mismatch)
            weight = idf_scores.get(word, 1.0)
            ##scaling the word's vector by its rarity before adding/they count more than common filler words
            vectors.append(model[word] * weight)
            weights.append(weight)
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
        ##weighted average instead of plain average- division by total weight instead of just the count of words
    return np.sum(vectors, axis=0) / np.sum(weights)

X_train_weighted = np.array([get_weighted_sentence_embedding(text, glove, idf_scores) for text in X_train_text])
X_test_weighted = np.array([get_weighted_sentence_embedding(text, glove, idf_scores) for text in X_test_text])

weighted_model = LogisticRegression(max_iter=1000)
weighted_model.fit(X_train_weighted, y_train_labels)
weighted_predictions = weighted_model.predict(X_test_weighted)

print("IDF-weighted embedding accuracy:", accuracy_score(y_test_labels, weighted_predictions))

IDF-weighted embedding accuracy: 0.5


## Takeaway

Instead of plain averaging, this weights each word's vector by how distinctive it is (its IDF score from Day 1) before averaging -- rare, meaningful words should count more than generic filler.

| Approach | Accuracy |
| --- | --- |
| CountVectorizer (Day 1) | 71.4% |
| TF-IDF (Day 1) | 62.5% |
| Plain averaged GloVe | 58.9% |
| IDF-weighted GloVe | 50.0% |

IDF-weighted averaging made things *worse*, not better -- 50% vs. plain averaging's 58.9%. This is a remarkably consistent pattern now, not a one-off: every time IDF-based weighting got added to this small dataset, it hurt accuracy. TF-IDF lost to CountVectorizer on Day 1, and now IDF-weighted embeddings just lost to plain-averaged embeddings. Same root cause both times: IDF weights are estimated from only 222 training documents, and with a corpus this small, "how rare is this word" is a genuinely noisy signal -- rare doesn't reliably mean meaningful here, it sometimes just means a one-off ticker symbol or unusual company name got overweighted.

Overall lesson for the day: pretrained embeddings are powerful for capturing real semantic relationships (the "stock"/"shares" similarity was genuinely impressive), but that doesn't automatically translate into a better classifier on a small, narrow, specific dataset like this one. Simple word-presence signals (CountVectorizer) keep winning here, and every attempt to add a smarter weighting scheme on top has made things worse, not better.